# Plant disease — Model B (field model)

**Deploy file:** `model_b_combined.keras` + `class_names.json`

Lab-only baseline (old Model A) is in a separate notebook: `train_model_a_kaggle.ipynb`.

Phase checkpoints (`best_model_b_p1/p1b/p2/p3.keras`) are training artifacts only.


In [ ]:
import os
import re
import json
import glob
import time
import hashlib
import shutil
import zipfile
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# --- Configuration (Model B only; optimized for field accuracy and Kaggle disk limits) ---
# Lab-only baseline (Model A) is in: notebooks/train_model_a_kaggle.ipynb
BACKBONE = 'efficientnetb0'
BUILD_MODEL = True               # True: train Model B; False: load an existing .keras model
DISK_SAFE_MODE = True            # Prevent large downloads (for example, 6.7GB PlantCity) into /working
INCLUDE_TOMATO_EXTRA = True
INCLUDE_PLANTCITY = 'auto'     # True: force download, False: skip, 'auto': use only Kaggle Input
MAX_FIELD_IMAGES_PER_FOLDER = 400  # Limit very large source folders (especially PlantCity)
FIELD_REPEAT = 16                # Field symlink/hardlink repeats (cheap; not full copies)
LAB_KEEP_FRACTION = 0.35         # Keep 35% of PlantVillage to reduce lab dominance
HARD_CLASS_EXTRA_REPEAT = 10     # Extra field repeats for weakest PlantDoc classes
PLANTDOC_VAL_FRACTION = 0.15     # PlantDoc hold-out fraction for early stopping
HEARTBEAT_EVERY_N_BATCHES = 50   # Minimal hang-safe: print progress during long epochs
USE_TTA = True
USE_ENSEMBLE = True              # Average predictions from best phase checkpoints during evaluation
USE_CLASS_WEIGHTS = True
USE_HARDLINKS = True             # Hardlink field images so raw directories can be removed safely
LABEL_SMOOTHING = 0.05
# Canonical deploy file (use this in Flask/production)
MODEL_PATH = '/kaggle/working/model_b_combined.keras'
BEST_PATH = '/kaggle/working/best_model_b.keras'
OLD_BUILD_DIRS = ['build', '/kaggle/input/plant-disease-build/build', '/kaggle/input/build']
WORKING_DIRS_TO_DROP_AFTER_LINK = []

# Weak classes from the latest PlantDoc run; give them extra field sampling weight
HARD_FIELD_CLASSES = {
    # Priority classes with low PlantDoc performance (but measurable on PlantDoc test)
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Northern_Leaf_Blight',
    'Soybean___healthy', 'Pepper,_bell___Bacterial_spot',
    'Tomato___Bacterial_spot', 'Tomato___Leaf_Mold', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___healthy', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Septoria_leaf_spot',
    'Cherry_(including_sour)___healthy', 'Grape___Black_rot',
}


def disk_status(label=''):
    u = shutil.disk_usage('/kaggle/working')
    free_gb, total_gb = u.free / 1e9, u.total / 1e9
    print(f'Disk [{label}]: {free_gb:.1f} GB free / {total_gb:.1f} GB total')
    return free_gb


def remove_zips_under(root_dir):
    if not os.path.isdir(root_dir):
        return
    for zp in glob.glob(os.path.join(root_dir, '**', '*.zip'), recursive=True):
        try:
            os.remove(zp)
            print(f'  removed zip: {os.path.basename(zp)}')
        except OSError:
            pass


def find_kaggle_input(*hints):
    base = '/kaggle/input'
    if not os.path.isdir(base):
        return None
    for root, dirs, files in os.walk(base):
        for d in dirs:
            low = d.lower()
            if any(h.lower() in low for h in hints):
                return os.path.join(root, d)
        # Limit depth to avoid scanning huge datasets if not found early
        depth = root[len(base):].count(os.sep)
        if depth >= 3:
            dirs.clear()
    return None


def link_file(src, dst):
    """Hardlink when possible; fall back to symlink (cross-mount). Returns 'hard'|'sym'."""
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.lexists(dst):
        return 'exists'
    if USE_HARDLINKS:
        try:
            os.link(src, dst)
            return 'hard'
        except OSError:
            pass
    os.symlink(os.path.abspath(src), dst)
    return 'sym'


def stable_bucket(key, mod=100):
    """Deterministic 0..mod-1 bucket (unlike Python's randomized hash())."""
    digest = hashlib.md5(key.encode('utf-8')).hexdigest()
    return int(digest[:8], 16) % mod


def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def collect_image_hashes(root):
    hashes = set()
    if not os.path.isdir(root):
        return hashes
    for dirpath, _, files in os.walk(root):
        for fname in files:
            low = fname.lower()
            if not low.endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.gif')):
                continue
            try:
                hashes.add(file_md5(os.path.join(dirpath, fname)))
            except OSError:
                pass
    return hashes


def safe_drop_working_dirs(dirs, combined_root):
    """Only delete raw downloads if links into combined_root are hardlinks that survive."""
    if not USE_HARDLINKS:
        print('Skipping raw-dir cleanup (USE_HARDLINKS=False — symlinks would break)')
        return
    sample = None
    if os.path.isdir(combined_root):
        for dirpath, _, files in os.walk(combined_root):
            for fname in files:
                p = os.path.join(dirpath, fname)
                if os.path.islink(p):
                    print(f'Skipping raw-dir cleanup — found symlink in {combined_root}: {fname}')
                    return
                if os.path.isfile(p):
                    sample = p
                    break
            if sample:
                break
    if sample is not None:
        try:
            if os.stat(sample).st_nlink < 2:
                print(f'Skipping raw-dir cleanup — sample link count={os.stat(sample).st_nlink} (expected hardlink)')
                return
        except OSError as e:
            print(f'Skipping raw-dir cleanup — cannot verify hardlinks: {e}')
            return
    for drop_dir in sorted(set(dirs)):
        if drop_dir.startswith('/kaggle/working/') and os.path.isdir(drop_dir):
            print(f'Freeing disk: removing raw field dir {drop_dir} (hardlinks kept)')
            shutil.rmtree(drop_dir, ignore_errors=True)

if BACKBONE == 'efficientnetb0':
    from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
elif BACKBONE == 'mobilenetv2':
    from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
else:
    raise ValueError(f'Unknown BACKBONE: {BACKBONE}')

# PlantDoc test support per class (0 means trained class, but not measurable on PlantDoc test)
PLANTDOC_TEST_SUPPORT = {
    'Apple___Apple_scab': 10, 'Apple___Black_rot': 0, 'Apple___Cedar_apple_rust': 10, 'Apple___healthy': 9,
    'Blueberry___healthy': 11, 'Cherry_(including_sour)___Powdery_mildew': 0, 'Cherry_(including_sour)___healthy': 10,
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 4, 'Corn_(maize)___Common_rust_': 10,
    'Corn_(maize)___Northern_Leaf_Blight': 12, 'Corn_(maize)___healthy': 0,
    'Grape___Black_rot': 8, 'Grape___Esca_(Black_Measles)': 0, 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 0,
    'Grape___healthy': 12, 'Orange___Haunglongbing_(Citrus_greening)': 0, 'Peach___Bacterial_spot': 0,
    'Peach___healthy': 9, 'Pepper,_bell___Bacterial_spot': 9, 'Pepper,_bell___healthy': 8,
    'Potato___Early_blight': 8, 'Potato___Late_blight': 8, 'Potato___healthy': 0,
    'Raspberry___healthy': 7, 'Soybean___healthy': 8, 'Squash___Powdery_mildew': 6,
    'Strawberry___Leaf_scorch': 0, 'Strawberry___healthy': 8, 'Tomato___Bacterial_spot': 9,
    'Tomato___Early_blight': 9, 'Tomato___Late_blight': 10, 'Tomato___Leaf_Mold': 6,
    'Tomato___Septoria_leaf_spot': 11, 'Tomato___Spider_mites Two-spotted_spider_mite': 0,
    'Tomato___Target_Spot': 0, 'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 6,
    'Tomato___Tomato_mosaic_virus': 10, 'Tomato___healthy': 8,
}
ZERO_TEST_CLASSES = [k for k, v in PLANTDOC_TEST_SUPPORT.items() if v == 0]

# --- Step 0: Dataset coverage reference (strengths and limitations by source) ---
DATASET_OVERVIEW = pd.DataFrame([
    {'Dataset': 'PlantVillage', 'Role': 'Lab images in Model B combined train', '~Images': '70k train',
     'Setting': 'Controlled lab', 'Classes': 'All 38',
     'Offers': 'Full label space, huge volume, ~96% lab acc',
     'Lacks': 'Real-world backgrounds; causes domain gap on PlantDoc'},
    {'Dataset': 'PlantDoc', 'Role': 'Field train + benchmark test', '~Images': '2.5k',
     'Setting': 'Real field photos', 'Classes': '27 of 38 in test',
     'Offers': 'Only honest real-world test set you have',
     'Lacks': '11 classes have zero test images; small vs PlantVillage'},
    {'Dataset': 'Tomato Multiple Sources', 'Role': 'Model B tomato boost', '~Images': '25k',
     'Setting': 'Lab + some wild', 'Classes': '10 tomato + healthy',
     'Offers': 'Mosaic, mites, target spot + foliar depth',
     'Lacks': 'Tomato only'},
    {'Dataset': 'PlantCity (Kaggle Input)', 'Role': 'Model B field boost', '~Images': '52k capped',
     'Setting': 'Pakistan field, smartphone', 'Classes': '52 (12 crops)',
     'Offers': 'Best PlantDoc-style field data — mount as Input (NOT downloaded to /working)',
     'Lacks': 'Add codewithsk/plantcity... in notebook Data tab'},
    {'Dataset': 'Removed (duplicate/small)', 'Role': '—', '~Images': '—',
     'Setting': '—', 'Classes': '—',
     'Offers': 'Orange 1.6k and Pakistan tomato 7.2k are already covered by selected sources',
     'Lacks': '—'},
])
print('=== Step 0: Dataset overview ===')
print(DATASET_OVERVIEW.to_string(index=False))
print(f'\nBackbone: {BACKBONE} | FIELD_REPEAT={FIELD_REPEAT} | LAB_KEEP={LAB_KEEP_FRACTION} | DISK_SAFE={DISK_SAFE_MODE}')
print(f'Field cap: {MAX_FIELD_IMAGES_PER_FOLDER}/folder | hardlinks={USE_HARDLINKS} | ensemble={USE_ENSEMBLE}')
print(f'Hard-class extra repeat: {HARD_CLASS_EXTRA_REPEAT} on {len(HARD_FIELD_CLASSES)} weak classes')
disk_status('start')
if DISK_SAFE_MODE and os.path.isdir('/kaggle/working/plantcity'):
    print('Removing stale /working/plantcity from a prior failed run...')
    shutil.rmtree('/kaggle/working/plantcity', ignore_errors=True)
    remove_zips_under('/kaggle/working')
    disk_status('after stale cleanup')
print(f'\nPlantDoc test: {sum(PLANTDOC_TEST_SUPPORT.values())} images across '
      f'{sum(1 for v in PLANTDOC_TEST_SUPPORT.values() if v > 0)} classes; '
      f'{len(ZERO_TEST_CLASSES)} classes have no test images (still trained, not scored in supported-F1)')
print('\n--- Field accuracy upgrades in this notebook ---')
print(f'1) {BACKBONE} backbone (stronger than MobileNetV2)')
print('2) Effective field sources only: PlantDoc + PlantCity (+ Tomato extra for tomato confusions)')
print('3) PlantDoc hold-out validation during training (early stopping on field)')
print('4) Field class weights (Phase 2 only) + field-only fine-tune')
print(f'5) TTA at eval (USE_TTA={USE_TTA})')
print('6) MD5 leakage filter (block PlantDoc train/test duplicates)')
print('7) Deterministic MD5 buckets for val split + folder caps')
print('8) Per-phase ModelCheckpoint (p1 / p1b / p2 / p3)')
print(f'9) Lab subsample {LAB_KEEP_FRACTION:.0%} + hard-class boost({HARD_CLASS_EXTRA_REPEAT}) + ensemble TTA')
print('10) Minimal hang-safe: HeartbeatCallback + timed save prints')

DATASET_OVERVIEW.to_csv('/kaggle/working/dataset_overview.csv', index=False)

# --- Step 1: PlantVillage (prefer Kaggle Input to save ~5GB in /working) ---
print('\n=== Step 1: PlantVillage ===')
PV_TRAIN = PV_VALID = None
pv_input = find_kaggle_input('new-plant-diseases', 'plant-diseases', 'plantvillage')
if pv_input:
    for root, dirs, _ in os.walk(pv_input):
        if 'train' in dirs and 'valid' in dirs:
            PV_TRAIN = os.path.join(root, 'train')
            PV_VALID = os.path.join(root, 'valid')
            print(f'Using PlantVillage from Kaggle Input: {root}')
            break
if PV_TRAIN is None:
    print('PlantVillage not in /kaggle/input — downloading to /working...')
    !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /kaggle/working/data --unzip
    remove_zips_under('/kaggle/working/data')
    PV_ROOT = '/kaggle/working/data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
    PV_TRAIN = os.path.join(PV_ROOT, 'train')
    PV_VALID = os.path.join(PV_ROOT, 'valid')
disk_status('after PlantVillage')

# --- Step 2: PlantDoc (real-field training source + benchmark test set) ---
print('=== Step 2: Download PlantDoc ===')
PD_DIR = '/kaggle/working/plantdoc'
if not os.path.exists(PD_DIR):
    !git clone -q https://github.com/pratikkayal/PlantDoc-Dataset.git {PD_DIR}
PD_TRAIN = os.path.join(PD_DIR, 'train')
PD_TEST = os.path.join(PD_DIR, 'test')

# --- Step 3: Extra field datasets (Model B only; keep PlantVillage from Step 1 intact) ---
print('=== Step 3: Extra field datasets ===')
EXTRA_TRAIN_SOURCES = []
tomato_extra_to_pv = {
    'Bacterial_spot': 'Tomato___Bacterial_spot', 'Early_blight': 'Tomato___Early_blight',
    'Late_blight': 'Tomato___Late_blight', 'Leaf_Mold': 'Tomato___Leaf_Mold',
    'Septoria_leaf_spot': 'Tomato___Septoria_leaf_spot',
    'Spider_mites Two-spotted_spider_mite': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'Target_Spot': 'Tomato___Target_Spot',
    'Tomato_Yellow_Leaf_Curl_Virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato_mosaic_virus': 'Tomato___Tomato_mosaic_virus', 'healthy': 'Tomato___healthy',
}

if INCLUDE_TOMATO_EXTRA:
    tom_input = find_kaggle_input('tomato-disease-multiple', 'tomato-disease')
    if tom_input and os.path.isdir(os.path.join(tom_input, 'train')):
        EXTRA_TRAIN_SOURCES.append((os.path.join(tom_input, 'train'), tomato_extra_to_pv, 'tms'))
        print(f'Tomato extra from Kaggle Input: {tom_input}')
    else:
        TOM_DIR = '/kaggle/working/tomato-extra'
        if not os.path.isdir(os.path.join(TOM_DIR, 'train')):
            if disk_status('before tomato download') < 2.0:
                print('Tomato extra skipped — not enough disk')
            else:
                print('Downloading Tomato Multiple Sources...')
                !kaggle datasets download -d cookiefinder/tomato-disease-multiple-sources -p {TOM_DIR} --unzip
                remove_zips_under(TOM_DIR)
        if os.path.isdir(os.path.join(TOM_DIR, 'train')):
            EXTRA_TRAIN_SOURCES.append((os.path.join(TOM_DIR, 'train'), tomato_extra_to_pv, 'tms'))
            WORKING_DIRS_TO_DROP_AFTER_LINK.append(TOM_DIR)

def find_class_folder_root(base_dir):
    """Find directory whose immediate children are class subfolders with images."""
    best, best_n = base_dir, 0
    for root, dirs, files in os.walk(base_dir):
        if len(dirs) > best_n or (len(dirs) == best_n and 'train' in root.lower() and 'test' not in root.lower()):
            best, best_n = root, len(dirs)
    return best

want_plantcity = INCLUDE_PLANTCITY is True or INCLUDE_PLANTCITY == 'auto'
if want_plantcity:
    pc_input = find_kaggle_input('plantcity')
    if pc_input:
        pc_root = find_class_folder_root(pc_input)
        EXTRA_TRAIN_SOURCES.append((pc_root, 'AUTO', 'pc'))
        print(f'PlantCity from Kaggle Input: {pc_root}')
    elif INCLUDE_PLANTCITY is True and not DISK_SAFE_MODE:
        if disk_status('before PlantCity') < 8.0:
            print('PlantCity skipped — need ~8GB free (add as Kaggle Input instead)')
        else:
            try:
                PC_DIR = '/kaggle/working/plantcity'
                if not os.path.isdir(PC_DIR) or not any(os.path.isdir(os.path.join(PC_DIR, d)) for d in os.listdir(PC_DIR)):
                    print('Downloading PlantCity (6.7GB)...')
                    !kaggle datasets download -d codewithsk/plantcity-a-comprehensive-images-multicrop-leaves -p {PC_DIR} --unzip
                    remove_zips_under(PC_DIR)
                pc_root = find_class_folder_root(PC_DIR)
                EXTRA_TRAIN_SOURCES.append((pc_root, 'AUTO', 'pc'))
                WORKING_DIRS_TO_DROP_AFTER_LINK.append(PC_DIR)
                print(f'PlantCity root: {pc_root}')
            except Exception as e:
                print(f'PlantCity skipped: {e}')
    else:
        print('PlantCity skipped — add dataset codewithsk/plantcity-a-comprehensive-images-multicrop-leaves as Kaggle Input')

print(f'Extra field sources for Model B: {len(EXTRA_TRAIN_SOURCES)}')
disk_status('after extra datasets')

# --- Step 4: Label harmonization maps (source labels -> PlantVillage 38 classes) ---
plantdoc_to_plantvillage = {
    'Apple Scab Leaf': 'Apple___Apple_scab', 'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': 'Apple___Cedar_apple_rust', 'Bell_pepper leaf': 'Pepper,_bell___healthy',
    'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot', 'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight', 'Corn rust leaf': 'Corn_(maize)___Common_rust_',
    'Peach leaf': 'Peach___healthy', 'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight', 'Raspberry leaf': 'Raspberry___healthy',
    'Soyabean leaf': 'Soybean___healthy', 'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf': 'Strawberry___healthy', 'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot', 'Tomato leaf': 'Tomato___healthy',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot', 'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf': 'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf': 'Grape___healthy', 'grape leaf black rot': 'Grape___Black_rot',
}
FIELD_ALIAS_MAP = {
    'apple scab': 'Apple___Apple_scab', 'apple black rot': 'Apple___Black_rot',
    'apple cedar apple rust': 'Apple___Cedar_apple_rust', 'apple cedar rust': 'Apple___Cedar_apple_rust',
    'apple healthy': 'Apple___healthy', 'blueberry healthy': 'Blueberry___healthy',
    'cherry powdery mildew': 'Cherry_(including_sour)___Powdery_mildew',
    'cherry healthy': 'Cherry_(including_sour)___healthy',
    'corn gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'corn cercospora leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'corn common rust': 'Corn_(maize)___Common_rust_',
    'corn northern leaf blight': 'Corn_(maize)___Northern_Leaf_Blight', 'corn healthy': 'Corn_(maize)___healthy',
    'grape black rot': 'Grape___Black_rot', 'grape esca': 'Grape___Esca_(Black_Measles)',
    'grape black measles': 'Grape___Esca_(Black_Measles)',
    'grape leaf blight': 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'grape isariopsis leaf spot': 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'grape healthy': 'Grape___healthy',
    'orange huanglongbing': 'Orange___Haunglongbing_(Citrus_greening)',
    'orange citrus greening': 'Orange___Haunglongbing_(Citrus_greening)',
    'citrus greening': 'Orange___Haunglongbing_(Citrus_greening)',
    'peach bacterial spot': 'Peach___Bacterial_spot', 'peach healthy': 'Peach___healthy',
    'bell pepper bacterial spot': 'Pepper,_bell___Bacterial_spot',
    'pepper bacterial spot': 'Pepper,_bell___Bacterial_spot',
    'bell pepper healthy': 'Pepper,_bell___healthy', 'pepper healthy': 'Pepper,_bell___healthy',
    'potato early blight': 'Potato___Early_blight', 'potato late blight': 'Potato___Late_blight',
    'potato healthy': 'Potato___healthy', 'raspberry healthy': 'Raspberry___healthy',
    'soybean healthy': 'Soybean___healthy', 'squash powdery mildew': 'Squash___Powdery_mildew',
    'strawberry leaf scorch': 'Strawberry___Leaf_scorch', 'strawberry healthy': 'Strawberry___healthy',
    'tomato bacterial spot': 'Tomato___Bacterial_spot', 'tomato early blight': 'Tomato___Early_blight',
    'tomato late blight': 'Tomato___Late_blight', 'tomato leaf mold': 'Tomato___Leaf_Mold',
    'tomato septoria leaf spot': 'Tomato___Septoria_leaf_spot',
    'tomato spider mites': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'tomato two spotted spider mite': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'tomato target spot': 'Tomato___Target_Spot',
    'tomato yellow leaf curl virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'tomato mosaic virus': 'Tomato___Tomato_mosaic_virus', 'tomato healthy': 'Tomato___healthy',
    'apple normal': 'Apple___healthy',
    'cherry normal leaf': 'Cherry_(including_sour)___healthy',
    'corn normal leaf': 'Corn_(maize)___healthy',
    'corn gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'grape normal leaf': 'Grape___healthy',
    'peach normal leaf': 'Peach___healthy',
    'potato normal leaf': 'Potato___healthy',
    'raspberry normal leaf': 'Raspberry___healthy',
    'soybean normal leaf': 'Soybean___healthy',
    'strawberry normal leaf': 'Strawberry___healthy',
    'tomato normal leaf': 'Tomato___healthy',
    'blueberry normal leaf': 'Blueberry___healthy',
    'pepper normal leaf': 'Pepper,_bell___healthy',
    'bell pepper normal leaf': 'Pepper,_bell___healthy',
}


def _norm_label(text):
    text = text.replace('+', ' ').replace('___', ' ')
    return ' '.join(re.sub(r'[^a-z0-9]+', ' ', text.lower()).split())


def _pv_tokens(pv_class):
    plant, disease = pv_class.split('___', 1)
    stop = {'including', 'sour', 'two', 'spotted', 'spider', 'mite', 'mites'}
    return set(_norm_label(plant.replace(',', '')).split()) | (set(_norm_label(disease).split()) - stop)


def build_auto_mapping(src_root, pv_classes, tag=''):
    pv_set = set(pv_classes)
    mapping = {}
    rows = []
    all_folders = [f for f in sorted(os.listdir(src_root)) if os.path.isdir(os.path.join(src_root, f))]
    for folder in all_folders:
        norm = _norm_label(folder)
        how = None
        if norm in FIELD_ALIAS_MAP and FIELD_ALIAS_MAP[norm] in pv_set:
            mapping[folder] = FIELD_ALIAS_MAP[norm]
            how = 'alias'
        else:
            tokens = set(norm.split())
            best, best_score = None, 0
            for pv in pv_classes:
                needed = _pv_tokens(pv)
                score = len(needed & tokens)
                if score > best_score and score >= max(2, len(needed) - 1):
                    best_score, best = score, pv
            if best:
                mapping[folder] = best
                how = f'token({best_score})'
        rows.append({
            'source': tag, 'folder': folder, 'norm': norm,
            'mapped_to': mapping.get(folder, ''), 'method': how or 'UNMAPPED',
        })
    print(f'  {tag}: mapped {len(mapping)} / {len(all_folders)} folders')
    out = f'/kaggle/working/mapping_{tag or "auto"}.csv'
    pd.DataFrame(rows).to_csv(out, index=False)
    print(f'  wrote {out}')
    unmapped = [r['folder'] for r in rows if not r['mapped_to']]
    if unmapped:
        print(f'  unmapped sample: {unmapped[:8]}')
    return mapping


def symlink_all(src_root, dst_root, keep_fraction=1.0):
    """Link lab images into dst. keep_fraction<1 stably subsamples to reduce lab dominance."""
    os.makedirs(dst_root, exist_ok=True)
    kept = skipped = 0
    keep_pct = int(max(0.0, min(1.0, keep_fraction)) * 100)
    for cls in os.listdir(src_root):
        src_dir = os.path.join(src_root, cls)
        if not os.path.isdir(src_dir):
            continue
        dst_dir = os.path.join(dst_root, cls)
        os.makedirs(dst_dir, exist_ok=True)
        for fname in os.listdir(src_dir):
            if keep_pct < 100 and stable_bucket(fname + '|lab|' + cls) >= keep_pct:
                skipped += 1
                continue
            src_path = os.path.abspath(os.path.join(src_dir, fname))
            dst = os.path.join(dst_dir, fname)
            if not os.path.lexists(dst):
                os.symlink(src_path, dst)
            kept += 1
    print(f'  lab links: kept={kept} skipped={skipped} (keep_fraction={keep_fraction})')


def relabel_into(src_root, dst_root, mapping, prefix, all_classes=None, repeat=1,
                 field_only_root=None, val_root=None, val_fraction=0.0,
                 max_per_folder=MAX_FIELD_IMAGES_PER_FOLDER, skip_hashes=None):
    if mapping == 'AUTO':
        mapping = build_auto_mapping(src_root, all_classes or [], tag=prefix)
    filter_leakage = skip_hashes is not None
    skip_hashes = skip_hashes or set()
    os.makedirs(dst_root, exist_ok=True)
    if all_classes:
        for cls in all_classes:
            os.makedirs(os.path.join(dst_root, cls), exist_ok=True)
    if field_only_root:
        os.makedirs(field_only_root, exist_ok=True)
        for cls in all_classes or []:
            os.makedirs(os.path.join(field_only_root, cls), exist_ok=True)
    if val_root:
        os.makedirs(val_root, exist_ok=True)
        for cls in all_classes or []:
            os.makedirs(os.path.join(val_root, cls), exist_ok=True)
    linked, val_linked, leaked = 0, 0, 0
    for folder in os.listdir(src_root):
        if folder not in mapping:
            continue
        cls = mapping[folder]
        fnames = [f for f in os.listdir(os.path.join(src_root, folder))
                  if os.path.isfile(os.path.join(src_root, folder, f))]
        if max_per_folder and len(fnames) > max_per_folder:
            fnames = sorted(fnames, key=lambda f: stable_bucket(f + '|' + folder, 10**9))[:max_per_folder]
        for fname in fnames:
            src_path = os.path.abspath(os.path.join(src_root, folder, fname))
            if skip_hashes:
                try:
                    if file_md5(src_path) in skip_hashes:
                        leaked += 1
                        continue
                except OSError:
                    pass
            to_val = val_root and stable_bucket(fname + '|' + folder) < int(val_fraction * 100)
            if to_val:
                vdst = os.path.join(val_root, cls, f'{prefix}_val_{fname}')
                link_file(src_path, vdst)
                val_linked += 1
                continue
            cls_repeat = repeat
            # Apply hard-class boost only to training; never modify held-out PlantDoc test distribution
            if field_only_root is not None and cls in HARD_FIELD_CLASSES:
                cls_repeat = repeat + HARD_CLASS_EXTRA_REPEAT
            for r in range(cls_repeat):
                tag = f'{prefix}_{r}_{fname}' if cls_repeat > 1 else f'{prefix}_{fname}'
                dst = os.path.join(dst_root, cls, tag)
                link_file(src_path, dst)
                if field_only_root:
                    fdst = os.path.join(field_only_root, cls, tag)
                    link_file(src_path, fdst)
                linked += 1
    cap_note = f', cap={max_per_folder}' if max_per_folder else ''
    extra = f', val={val_linked}' if val_root else ''
    leak_note = f', leaked_skipped={leaked}' if filter_leakage else ''
    hard_note = f', hard_extra={HARD_CLASS_EXTRA_REPEAT}' if field_only_root is not None else ''
    print(f'  {prefix}: {linked} train links (base_repeat={repeat}{hard_note}{cap_note}{extra}{leak_note})')


# --- Step 5: Build training directories (all 38 classes; lab + boosted field) ---
print('=== Step 5: Build training sets ===')
class_names = sorted([d for d in os.listdir(PV_TRAIN) if os.path.isdir(os.path.join(PV_TRAIN, d))])
NUM_CLASSES = len(class_names)
assert NUM_CLASSES == 38, f'Expected 38 classes, got {NUM_CLASSES}'

# Hash PlantDoc test images first, then skip byte-identical train images to prevent leakage
print('Hashing PlantDoc test images to block train/test leakage...')
PD_TEST_HASHES = collect_image_hashes(PD_TEST)
print(f'PlantDoc test unique MD5s: {len(PD_TEST_HASHES)}')

COMBINED_TRAIN = '/kaggle/working/combined-train'
FIELD_ONLY_TRAIN = '/kaggle/working/field-only-train'
PD_VAL_RELABELED = '/kaggle/working/plantdoc-val-relabeled'
PD_TEST_RELABELED = '/kaggle/working/plantdoc-test-relabeled'
for d in [COMBINED_TRAIN, FIELD_ONLY_TRAIN, PD_VAL_RELABELED, PD_TEST_RELABELED]:
    if os.path.exists(d):
        shutil.rmtree(d)

symlink_all(PV_TRAIN, COMBINED_TRAIN, keep_fraction=LAB_KEEP_FRACTION)
relabel_into(PD_TRAIN, COMBINED_TRAIN, plantdoc_to_plantvillage, 'pd', class_names,
              repeat=FIELD_REPEAT, field_only_root=FIELD_ONLY_TRAIN,
              val_root=PD_VAL_RELABELED, val_fraction=PLANTDOC_VAL_FRACTION,
              skip_hashes=PD_TEST_HASHES)
for src_root, mapping, prefix in EXTRA_TRAIN_SOURCES:
    relabel_into(src_root, COMBINED_TRAIN, mapping, prefix, class_names,
                  repeat=FIELD_REPEAT, field_only_root=FIELD_ONLY_TRAIN,
                  skip_hashes=PD_TEST_HASHES)

safe_drop_working_dirs(WORKING_DIRS_TO_DROP_AFTER_LINK, COMBINED_TRAIN)
remove_zips_under('/kaggle/working')
disk_status('after build + cleanup')

lab_n = sum(len(os.listdir(os.path.join(PV_TRAIN, c))) for c in class_names)
combo_n = sum(len(os.listdir(os.path.join(COMBINED_TRAIN, c))) for c in class_names)
field_n = sum(len(os.listdir(os.path.join(FIELD_ONLY_TRAIN, c))) for c in class_names
              if os.path.isdir(os.path.join(FIELD_ONLY_TRAIN, c)))
print(f'All {NUM_CLASSES} classes kept | Lab: {lab_n} | Combined: {combo_n} | Field pool: {field_n}')

relabel_into(PD_TEST, PD_TEST_RELABELED, plantdoc_to_plantvillage, 'pd', class_names)

# Recompute PlantDoc test support from the cloned data (overrides hardcoded values if different)
live_support = {}
for c in class_names:
    cdir = os.path.join(PD_TEST_RELABELED, c)
    live_support[c] = len(os.listdir(cdir)) if os.path.isdir(cdir) else 0
drift = [c for c in class_names if live_support[c] != PLANTDOC_TEST_SUPPORT.get(c, 0)]
if drift:
    print(f'PlantDoc support drift vs hardcoded dict ({len(drift)} classes):')
    for c in drift[:12]:
        print(f'  {c}: hardcoded={PLANTDOC_TEST_SUPPORT.get(c, 0)} live={live_support[c]}')
PLANTDOC_TEST_SUPPORT = live_support
ZERO_TEST_CLASSES = [k for k, v in PLANTDOC_TEST_SUPPORT.items() if v == 0]

with open('/kaggle/working/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

coverage_df = pd.DataFrame({
    'Class': class_names,
    'PlantDoc Test Support': [PLANTDOC_TEST_SUPPORT.get(c, 0) for c in class_names],
    'Measurable on PlantDoc': ['Yes' if PLANTDOC_TEST_SUPPORT.get(c, 0) > 0 else 'No (train only)'
                               for c in class_names],
})
coverage_df.to_csv('/kaggle/working/class_coverage.csv', index=False)

# --- Step 6: Input pipelines ---
IMG_SIZE, BATCH = (224, 224), 32
aug_lab = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])
aug_field = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.25),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.25),
    tf.keras.layers.RandomContrast(0.3),
])


def make_ds(directory, shuffle=True, strong_aug=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory, image_size=IMG_SIZE, batch_size=BATCH, shuffle=shuffle,
        seed=42, class_names=class_names)
    if shuffle:
        aug = aug_field if strong_aug else aug_lab
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


combined_train_ds = make_ds(COMBINED_TRAIN, strong_aug=True)
field_only_ds = make_ds(FIELD_ONLY_TRAIN, strong_aug=True)
pv_valid_ds = make_ds(PV_VALID, shuffle=False)
pd_val_ds = make_ds(PD_VAL_RELABELED, shuffle=False)
pd_test_ds = make_ds(PD_TEST_RELABELED, shuffle=False)

# Build class weights from field counts; apply only in Phase 2 (field-only fine-tuning)
field_class_weight = None
if USE_CLASS_WEIGHTS:
    field_counts = []
    for c in class_names:
        fdir = os.path.join(FIELD_ONLY_TRAIN, c)
        field_counts.append(len(os.listdir(fdir)) if os.path.isdir(fdir) else 1)
    max_f = max(field_counts)
    field_class_weight = {}
    for i, cnt in enumerate(field_counts):
        w = min(6.0, max_f / max(cnt, 1))
        if class_names[i] in HARD_FIELD_CLASSES:
            w = min(8.0, w * 1.75)
        field_class_weight[i] = float(w)
    print(f'Field class weights (Phase 2 only): min={min(field_class_weight.values()):.2f}, max={max(field_class_weight.values()):.2f}')
    hard_ws = [(class_names[i], field_class_weight[i]) for i in range(NUM_CLASSES) if class_names[i] in HARD_FIELD_CLASSES]
    print('  hard-class weights:', {k: round(v, 2) for k, v in hard_ws})

# --- Step 7: Train Model B ---
def sparse_ce(label_smoothing=0.0):
    if label_smoothing and label_smoothing > 0:
        def _loss(y_true, y_pred):
            y = tf.one_hot(tf.cast(y_true, tf.int32), NUM_CLASSES)
            return tf.keras.losses.categorical_crossentropy(y, y_pred, label_smoothing=label_smoothing)
        _loss.__name__ = 'sparse_ce_smooth'
        return _loss
    return 'sparse_categorical_crossentropy'


class HeartbeatCallback(tf.keras.callbacks.Callback):
    """Minimal hang-safe: periodic batch log so a silent progress bar is not mistaken for a dead run."""
    def __init__(self, every_n=50):
        super().__init__()
        self.every_n = max(1, int(every_n))
        self._t0 = None

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.time()
        print(f'  [heartbeat] epoch {epoch + 1} started', flush=True)

    def on_train_batch_end(self, batch, logs=None):
        if batch == 0 or (batch + 1) % self.every_n == 0:
            logs = logs or {}
            loss = logs.get('loss')
            loss_s = f'{float(loss):.4f}' if loss is not None else 'n/a'
            elapsed = time.time() - self._t0 if self._t0 else 0.0
            print(f'  [heartbeat] batch {batch + 1} loss={loss_s} elapsed={elapsed:.0f}s', flush=True)

    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self._t0 if self._t0 else 0.0
        print(
            f'  [heartbeat] epoch {epoch + 1} train done in {elapsed:.0f}s '
            f'— running val/ckpt (full .keras save can take a few minutes)...',
            flush=True,
        )


def timed_save(model, path, label=''):
    """Print timing around model.save so long writes are not mistaken for hangs."""
    label = label or path
    print(f'  saving {label}...', flush=True)
    t0 = time.time()
    model.save(path)
    print(f'  saved {path} ({time.time() - t0:.1f}s)', flush=True)


def build_model():
    if BACKBONE == 'efficientnetb0':
        base = EfficientNetB0(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
        fine_tune_at = len(base.layers) - 80  # Unfreeze deeper layers for stronger field adaptation
    else:
        base = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
        fine_tune_at = len(base.layers) - 40
    base.trainable = False
    model = tf.keras.Sequential([
        base, tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(512, activation='relu'), tf.keras.layers.Dropout(0.45),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss=sparse_ce(0.0), metrics=['accuracy'])
    return model, base, fine_tune_at


def field_callbacks(tag, patience=3):
    """Fresh EarlyStopping + ModelCheckpoint per phase (Checkpoint.best is not reset across fit())."""
    path = f'/kaggle/working/best_{tag}.keras'
    return path, [
        HeartbeatCallback(HEARTBEAT_EVERY_N_BATCHES),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=patience, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=max(1, patience - 1), min_lr=1e-7),
        tf.keras.callbacks.ModelCheckpoint(
            path, monitor='val_accuracy', save_best_only=True, save_weights_only=False),
    ]


def _load_best(model, path):
    if os.path.isfile(path):
        print(f'  reloading best weights from {path}', flush=True)
        return tf.keras.models.load_model(path, compile=False)
    print(f'  no checkpoint at {path}; keeping current weights', flush=True)
    return model


def train_model():
    model, base, fine_tune_at = build_model()
    cw_field = field_class_weight if USE_CLASS_WEIGHTS else None

    print('Phase 1: subsampled lab + boosted field (PlantDoc val)...')
    path1, cbs1 = field_callbacks('model_b_p1', patience=3)
    model.fit(combined_train_ds, validation_data=pd_val_ds, epochs=10, callbacks=cbs1)
    model = _load_best(model, path1)
    # Rebind backbone reference after reload (in Sequential, layer 0 is the backbone)
    base = model.layers[0]

    base.trainable = True
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss=sparse_ce(LABEL_SMOOTHING), metrics=['accuracy'])
    print('Phase 1b: fine-tune backbone on combined...')
    path1b, cbs1b = field_callbacks('model_b_p1b', patience=3)
    model.fit(combined_train_ds, validation_data=pd_val_ds, epochs=6, callbacks=cbs1b)
    model = _load_best(model, path1b)
    base = model.layers[0]

    base.trainable = True
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-6),
                  loss=sparse_ce(LABEL_SMOOTHING), metrics=['accuracy'])
    print('Phase 2: field-only fine-tune (class weights + hard-class boost)...')
    path2, cbs2 = field_callbacks('model_b_p2', patience=4)
    model.fit(field_only_ds, validation_data=pd_val_ds, epochs=10,
              class_weight=cw_field, callbacks=cbs2)
    model = _load_best(model, path2)

    # Phase 3: short mixed revisit to retain lab robustness while still selecting on PlantDoc val
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-6),
                  loss=sparse_ce(LABEL_SMOOTHING), metrics=['accuracy'])
    print('Phase 3: light mixed revisit on combined...')
    path3, cbs3 = field_callbacks('model_b_p3', patience=2)
    model.fit(combined_train_ds, validation_data=pd_val_ds, epochs=3, callbacks=cbs3)
    model = _load_best(model, path3)

    timed_save(model, BEST_PATH, label='canonical best')
    return model


print('=== Step 7: Train Model B (field model) ===')
print(f'BUILD_MODEL={BUILD_MODEL}  deploy_path={MODEL_PATH}')

model = None
if BUILD_MODEL:
    model = train_model()
    timed_save(model, MODEL_PATH, label='deploy Model B')
    print(f'Saved Model B (deploy) -> {MODEL_PATH}')
elif os.path.isfile(MODEL_PATH):
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print(f'Loaded Model B from {MODEL_PATH}')
elif os.path.isfile(BEST_PATH):
    model = tf.keras.models.load_model(BEST_PATH, compile=False)
    print(f'Loaded Model B from {BEST_PATH}')
else:
    raise RuntimeError(
        'Model B not available. Set BUILD_MODEL=True to train, or place weights at '
        f'{MODEL_PATH} or {BEST_PATH}')

assert model is not None, 'Model B is required'
model_b = model  # Keep alias for report keys and existing plotting code

# --- Step 8: Evaluate (supported-class F1 excludes classes with zero PlantDoc test support) ---
def tta_predict_one(model, imgs):
    """Average predictions over flip + central-crop views (imgs already preprocessed)."""
    # predict_on_batch avoids Dataset-based predict() overhead/leaks in long eval loops
    x = imgs.numpy() if hasattr(imgs, 'numpy') else np.asarray(imgs)
    preds = [model.predict_on_batch(x)]
    if USE_TTA:
        preds.append(model.predict_on_batch(np.flip(x, axis=2)))  # left-right
        preds.append(model.predict_on_batch(np.flip(x, axis=1)))  # up-down
        preds.append(model.predict_on_batch(np.flip(np.flip(x, axis=1), axis=2)))
        # central crops via TF then back to numpy for predict_on_batch
        xt = tf.convert_to_tensor(x)
        crop95 = tf.image.resize(tf.image.central_crop(xt, 0.95), IMG_SIZE).numpy()
        crop90 = tf.image.resize(tf.image.central_crop(xt, 0.90), IMG_SIZE).numpy()
        preds.append(model.predict_on_batch(crop95))
        preds.append(model.predict_on_batch(crop90))
    return sum(preds) / float(len(preds))


def batch_predict(models, imgs):
    if not isinstance(models, (list, tuple)):
        models = [models]
    return sum(tta_predict_one(m, imgs) for m in models) / float(len(models))


def predict_all(models, dataset):
    y_true, y_pred = [], []
    for imgs, labels in dataset:
        preds = batch_predict(models, imgs)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
    return np.array(y_true), np.array(y_pred)


def load_ensemble_models(primary):
    if not USE_ENSEMBLE:
        return [primary]
    paths, seen = [], set()
    for p in [
        '/kaggle/working/best_model_b_p2.keras',
        '/kaggle/working/best_model_b_p3.keras',
        '/kaggle/working/best_model_b_p1b.keras',
        BEST_PATH,
        MODEL_PATH,
    ]:
        if os.path.isfile(p) and p not in seen:
            paths.append(p)
            seen.add(p)
        if len(paths) >= 2:
            break
    if not paths:
        return [primary]
    print('Ensemble models (eval only):', paths)
    return [tf.keras.models.load_model(p, compile=False) for p in paths]


supported_idx = [class_names.index(c) for c in class_names if PLANTDOC_TEST_SUPPORT.get(c, 0) > 0]
TOMATO_FOLIAR = [
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
]
TOMATO_IDX = [class_names.index(c) for c in TOMATO_FOLIAR]

print('=== Step 8: Evaluate Model B ===')
eval_models = load_ensemble_models(model_b)
model_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
_, pv_acc = model_b.evaluate(pv_valid_ds, verbose=0)
y_true, y_pred = predict_all(eval_models, pd_test_ds)
pd_acc = float(np.mean(y_true == y_pred))
_, _, macro_all, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(NUM_CLASSES)), average='macro', zero_division=0)
_, _, macro_sup, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=supported_idx, average='macro', zero_division=0)
_, _, weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
results = {
    'Model B': dict(pv_acc=float(pv_acc), pd_acc=float(pd_acc),
                    macro_f1_all=float(macro_all), macro_f1_supported=float(macro_sup),
                    weighted_f1=float(weighted), y_true=y_true, y_pred=y_pred)
}

# --- Step 9: Generate reports ---
print('\n=== Step 9: Results ===')
summary = pd.DataFrame([
    {'Model': k, 'PV Acc': f"{v['pv_acc']*100:.1f}%", 'PlantDoc Acc': f"{v['pd_acc']*100:.1f}%",
     'Macro F1 (27 supported)': f"{v['macro_f1_supported']:.3f}",
     'Macro F1 (all 38)': f"{v['macro_f1_all']:.3f}", 'Weighted F1': f"{v['weighted_f1']:.3f}"}
    for k, v in results.items()
])
print(summary.to_string(index=False))
print(f"\nModel B PlantDoc acc: {results['Model B']['pd_acc']*100:.1f}%")
print(f'Deploy file: {MODEL_PATH}')
print(f'Headline metric: Macro F1 (27 supported) — fair score excluding {len(ZERO_TEST_CLASSES)} classes with no PlantDoc test')

for d in OLD_BUILD_DIRS:
    p = os.path.join(d, 'evaluation_report.json')
    if os.path.isfile(p):
        old = json.load(open(p))['summary']['model_b']
        print(f"Old build B: {old['pd_acc']*100:.1f}% PlantDoc, macro-F1(all-38)={old.get('pd_macro_f1', 0):.3f}")
        break

_, _, b_f1, _ = precision_recall_fscore_support(
    results['Model B']['y_true'], results['Model B']['y_pred'],
    labels=list(range(NUM_CLASSES)), zero_division=0)
per_class = pd.DataFrame({
    'Class': class_names,
    'PlantDoc Support': [PLANTDOC_TEST_SUPPORT.get(c, 0) for c in class_names],
    'B F1': np.round(b_f1, 3),
})
per_class.to_csv('/kaggle/working/per_class_metrics.csv', index=False)

measurable = per_class[per_class['PlantDoc Support'] > 0].sort_values('B F1', ascending=False)
print('\nTop 5 (measurable classes):')
print(measurable.head(5).to_string(index=False))
print('\nBottom 5 (measurable classes):')
print(measurable.tail(5).to_string(index=False))
print('\nTrain-only (no PlantDoc test — still in model):')
print(per_class[per_class['PlantDoc Support'] == 0]['Class'].tolist())

mask = np.isin(results['Model B']['y_true'], TOMATO_IDX)
yt, yp = results['Model B']['y_true'][mask], results['Model B']['y_pred'][mask]
cm = confusion_matrix(yt, yp, labels=TOMATO_IDX)
pd.DataFrame(cm, index=TOMATO_FOLIAR, columns=TOMATO_FOLIAR).to_csv('/kaggle/working/tomato_foliar_b.csv')

report = {
    'num_classes': NUM_CLASSES,
    'zero_test_classes': ZERO_TEST_CLASSES,
    'summary': {k: {kk: v[kk] for kk in ('pv_acc', 'pd_acc', 'macro_f1_all', 'macro_f1_supported', 'weighted_f1')}
                for k, v in results.items()},
    'per_class': per_class.to_dict('records'),
}
with open('/kaggle/working/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

plot_models = ['Model B']
fig, axes = plt.subplots(1, 1, figsize=(12, 9))
axes = [axes]
for ax, key in zip(axes, plot_models):
    cm = confusion_matrix(results[key]['y_true'], results[key]['y_pred'], labels=range(NUM_CLASSES))
    sns.heatmap(cm, ax=ax, cmap='Blues', xticklabels=class_names, yticklabels=class_names, cbar=False)
    ax.set_title(key)
    ax.tick_params(labelsize=5)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrices.png', dpi=200)
plt.close()

y = np.arange(len(measurable))
plt.figure(figsize=(11, 9))
plt.barh(y, measurable['B F1'], 0.4, label='Model B')
plt.yticks(y, measurable['Class'], fontsize=7)
plt.xlabel('F1')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/per_class_f1.png', dpi=200)
plt.close()

x = ['PlantVillage', 'PlantDoc']
plt.figure(figsize=(8, 5))
w = 0.5
colors = {'Model B': '#4C72B0'}
for i, label in enumerate(plot_models):
    vals = [results[label]['pv_acc'] * 100, results[label]['pd_acc'] * 100]
    bars = plt.bar(np.arange(2), vals, w, label=label, color=colors[label])
    for b in bars:
        plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 1, f'{b.get_height():.1f}%', ha='center', fontsize=9)
plt.xticks(range(2), x)
plt.ylim(0, 105)
plt.ylabel('Accuracy %')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/domain_gap.png', dpi=200)
plt.close()
print('\nSaved: dataset_overview.csv, class_coverage.csv, evaluation_report.json, per_class_metrics.csv, plots')

# --- Step 10: Slim Kaggle output (keep only models + reports; remove datasets) ---
# Kaggle Save & Commit packages everything in /kaggle/working, so remove heavy directories first.
KEEP_NAMES = {
    # Production model used by the app:
    'model_b_combined.keras',
    # Training checkpoints (for analysis/recovery, not production):
    'best_model_b.keras', 'best_model_b_p1.keras', 'best_model_b_p1b.keras',
    'best_model_b_p2.keras', 'best_model_b_p3.keras', 'dsn_artifacts.zip',
    'class_names.json', 'evaluation_report.json', 'per_class_metrics.csv',
    'class_coverage.csv', 'dataset_overview.csv', 'tomato_foliar_b.csv',
    'confusion_matrices.png', 'per_class_f1.png', 'domain_gap.png',
}
KEEP_PREFIXES = ('mapping_',)

DROP_DIRS = [
    'data', 'plantdoc', 'tomato-extra', 'plantcity', 'pk-tomato', 'orange-extra',
    'combined-train', 'field-only-train',
    'plantdoc-val-relabeled', 'plantdoc-test-relabeled',
]

print('\n=== Step 10: Slim output for Save & Commit ===')
disk_status('before slim')

for name in DROP_DIRS:
    p = os.path.join('/kaggle/working', name)
    if os.path.isdir(p):
        print(f'  removing {name}/')
        shutil.rmtree(p, ignore_errors=True)

# Remove leftover archives and temporary files; keep only intended artifacts
for entry in os.listdir('/kaggle/working'):
    p = os.path.join('/kaggle/working', entry)
    if entry in KEEP_NAMES or any(entry.startswith(pref) for pref in KEEP_PREFIXES):
        continue
    if entry.endswith('.zip') and entry != 'dsn_artifacts.zip':
        try:
            os.remove(p)
            print(f'  removed {entry}')
        except OSError:
            pass
        continue
    if os.path.isdir(p):
        print(f'  removing leftover dir {entry}/')
        shutil.rmtree(p, ignore_errors=True)
    elif os.path.isfile(p) and entry not in KEEP_NAMES:
        # Keep expected notebook outputs; remove unknown large files
        if entry.endswith(('.zip', '.tar', '.gz', '.jpg', '.jpeg', '.png', '.h5', '.keras')) and entry not in KEEP_NAMES:
            try:
                os.remove(p)
                print(f'  removed {entry}')
            except OSError:
                pass

# Build a compact zip for one-click download (models + reports only)
ART_ZIP = '/kaggle/working/dsn_artifacts.zip'
with zipfile.ZipFile(ART_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for entry in sorted(os.listdir('/kaggle/working')):
        if entry == 'dsn_artifacts.zip':
            continue
        if entry in KEEP_NAMES or any(entry.startswith(pref) for pref in KEEP_PREFIXES):
            p = os.path.join('/kaggle/working', entry)
            if os.path.isfile(p):
                zf.write(p, arcname=entry)
                print(f'  packed {entry} ({os.path.getsize(p)/1e6:.1f} MB)')

disk_status('after slim')
print(f'\nDownload this from Output: dsn_artifacts.zip ({os.path.getsize(ART_ZIP)/1e6:.1f} MB)')
print('Save & Commit will now package models + reports only (datasets deleted from /working).')
